## LanceDB Vector Database

In [3]:
#!pip install lancedb

### Connect to a database

In [4]:
import lancedb
import pandas as pd
import pyarrow as pa

uri = "data/sample-lancedb"
db = lancedb.connect(uri)

# LanceDb offers both a synchronous client. 
# In this guide we will give examples of synchronous clients.  
#uri = "data/sample-lancedb"
#async_db = await lancedb.connect_async(uri)

In [5]:
db.uri

'/home/maulik/CS104/Week 14 - Embeddings & Vector DBs/notebooks_dataset/data/sample-lancedb'

### Create a table
Create a table from initial data.
If you have data to insert into the table at creation time, you can simultaneously create a table and insert the data into it. The schema of the data will be used as the schema of the table.

In [10]:
data = [
    {"vector": [3.1, 4.1], "item": "foo", "price": 10.0},
    {"vector": [5.9, 26.5], "item": "bar", "price": 20.0},
]

# Synchronous client
tbl = db.create_table("my_table", data=data, mode="overwrite")

# Perform a quick vector search to verify
results = tbl.search([3.0, 4.0]).limit(1).to_list()
print("Search Results:", results)

Search Results: [{'vector': [3.0999999046325684, 4.099999904632568], 'item': 'foo', 'price': 10.0, '_distance': 0.01999996230006218}]


### Create an empty table
Sometimes you may not have the data to insert into the table at creation time. In this case, you can create an empty table and specify the schema, so that you can add data to the table at a later time (as long as it conforms to the schema). This is similar to a CREATE TABLE statement in SQL.

In [12]:
# Connect to the local database
db = lancedb.connect("data/sample-lancedb")

schema = pa.schema([pa.field("vector", pa.list_(pa.float32(), list_size=2))])
# Synchronous client
tbl = db.create_table("empty_table3", schema=schema, mode="overwrite")
print(f"Table created. Row count: {tbl.count_rows()}")

Table created. Row count: 0


### Open an existing table
Once created, you can open a table as follows

In [14]:
# Synchronous client
tbl = db.open_table("my_table")
# Asynchronous client
#async_tbl = await async_db.open_table("my_table2")

If you forget the name of your table, you can always get a listing of all table names:

In [15]:
# Synchronous client
print(db.table_names())
# Asynchronous client
#print(await async_db.table_names())

['empty_table3', 'my_table', 'my_vectors']


/tmp/ipykernel_780/1963261713.py:2: DeprecationWarning: table_names() is deprecated, use list_tables() instead
  print(db.table_names())


### Add data to a table
After a table has been created, you can always add more data to it as follows:

In [16]:
# Option 1: Add a list of dicts to a table
data = [
    {"vector": [1.3, 1.4], "item": "fizz", "price": 100.0},
    {"vector": [9.5, 56.2], "item": "buzz", "price": 200.0},
]
tbl.add(data)

# Option 2: Add a pandas DataFrame to a table
df = pd.DataFrame(data)
tbl.add(data)
# Asynchronous client
# await async_tbl.add(data)

AddResult(version=12)

### Note on Creating Tables
If the table already exists, LanceDB will raise an error by default.

create_table supports an optional exist_ok parameter. When set to True and the table exists, then it simply opens the existing table. The data you passed in will NOT be appended to the table in that case.

In [17]:
db.create_table("my_vectors", data, exist_ok=True)

LanceTable(name='my_vectors', _conn=LanceDBConnection(uri='/home/maulik/CS104/Week 14 - Embeddings & Vector DBs/notebooks_dataset/data/sample-lancedb'))

Sometimes you want to make sure that you start fresh. If you want to overwrite the table, you can pass in mode="overwrite" to the createTable function.


In [18]:
db.create_table("my_vectors", data, mode="overwrite")

LanceTable(name='my_vectors', _conn=LanceDBConnection(uri='/home/maulik/CS104/Week 14 - Embeddings & Vector DBs/notebooks_dataset/data/sample-lancedb'))

### Adding a Panda to the table
After a table has been created, you can always add more data to it using the various methods available. 
You can add any of the valid data structures accepted by LanceDB table, i.e, `dict`, `list[dict]`, `pd.DataFrame`, or `Iterator[pa.RecordBatch]`. Below are some examples.

In [19]:
df = pd.DataFrame({
    "vector": [[1.3, 1.4], [9.5, 56.2]], "item": ["banana", "apple"], "price": [5.0, 7.0]
})
df.head

<bound method NDFrame.head of         vector    item  price
0   [1.3, 1.4]  banana    5.0
1  [9.5, 56.2]   apple    7.0>

In [20]:
tbl.add(df)

AddResult(version=13)

### Add an Iterator
You can also add a large dataset batch in one go using Iterator of any supported data types.

In [21]:
def make_batches():
    for i in range(3):
        yield [
                {"vector": [3.1, 4.1], "item": "peach", "price": 6.0},
                {"vector": [5.9, 26.5], "item": "pear", "price": 5.0},
                {"vector": [5.1, 20.5], "item": "plum", "price": 4.0}
            ]
tbl.add(make_batches())

AddResult(version=14)

### Search is Select
You can select and filter data with `search()`

In [22]:
tbl.search().limit(6).to_pandas()

,vector,item,price
0,"[3.1, 4.1]",foo,10.0
1,"[5.9, 26.5]",bar,20.0
2,"[1.3, 1.4]",fizz,100.0
3,"[9.5, 56.2]",buzz,200.0
4,"[1.3, 1.4]",fizz,100.0
5,"[9.5, 56.2]",buzz,200.0


In [23]:
tbl.search().limit(3).to_arrow()

pyarrow.Table
vector: fixed_size_list<item: float>[2]
  child 0, item: float
item: string
price: double
----
vector: [[[3.1,4.1],[5.9,26.5]],[[1.3,1.4]]]
item: [["foo","bar"],["fizz"]]
price: [[10,20],[100]]

In [24]:
tbl.search().where("price = 10").limit(10).to_pandas()

,vector,item,price
0,"[3.1, 4.1]",foo,10.0


In [25]:
tbl.search().where("price = 10.0").limit(10).to_pandas()

,vector,item,price
0,"[3.1, 4.1]",foo,10.0


In [26]:
#dir(async_tbl)

In [27]:
tbl.count_rows()

17

### Filters
#### Pre and post-filtering
LanceDB supports filtering of query results based on metadata fields. By default, post-filtering is performed on the top-k results returned by the vector search. However, pre-filtering is also an option that performs the filter prior to vector search. This can be useful to narrow down on the search space on a very large dataset to reduce query latency


In [28]:
result = (
    tbl.search()
    .where("item =='peach'", prefilter=True)
    .limit(92)
    .to_pandas()
)

In [29]:
result

,vector,item,price
0,"[3.1, 4.1]",peach,6.0
1,"[3.1, 4.1]",peach,6.0
2,"[3.1, 4.1]",peach,6.0


### Deleting from a table
Use the `delete()` method on tables to delete rows from a table. To choose which rows to delete, provide a filter that matches on the metadata columns. This can delete any number of rows that match the filter.

In [30]:
tbl.count_rows()

17

In [31]:
tbl.delete('item = "peach"')

DeleteResult(num_deleted_rows=3, version=15)

In [32]:
tbl.count_rows()

14

In [33]:
to_remove =[5.0,7.0]
to_remove =",".join(str(v) for v in to_remove)
tbl.delete(f"price IN ({to_remove})")

DeleteResult(num_deleted_rows=5, version=16)

In [34]:
tbl.search().to_pandas()

,vector,item,price
0,"[3.1, 4.1]",foo,10.0
1,"[5.9, 26.5]",bar,20.0
2,"[1.3, 1.4]",fizz,100.0
3,"[9.5, 56.2]",buzz,200.0
4,"[1.3, 1.4]",fizz,100.0
5,"[9.5, 56.2]",buzz,200.0
6,"[5.1, 20.5]",plum,4.0
7,"[5.1, 20.5]",plum,4.0
8,"[5.1, 20.5]",plum,4.0


### Updating a table
You can `update` zero to all rows depending on how many rows match the `where` clause. The update queries follow the form of a SQL UPDATE statement. The `where` parameter is a SQL filter that matches on the metadata columns. The `values` or `values_sql` parameters are used to provide the new values for the columns.

In [35]:
import lancedb
import pandas as pd

# Create a lancedb connection
db = lancedb.connect("./.lancedb")

# Create a table from a pandas DataFrame
data = pd.DataFrame({"x": [1, 2, 3], "vector": [[1, 2], [3, 4], [5, 6]]})
table = db.create_table("my_table", data)

# Update the table where x = 2
table.update(where="x = 2", values={"vector": [10, 10]})

# Get the updated table as a pandas DataFrame
df = table.to_pandas()

# Print the DataFrame
print(df)

   x    vector
0  1    [1, 2]
1  3    [5, 6]
2  2  [10, 10]


The `values` parameter is used to provide the new values for the columns as literal values. You can also use the `values_sql / valuesSql` parameter to provide SQL expressions for the new values. For example, you can use `values_sql="x + 1"` to increment the value of the `x` column by `1`.

In [36]:
# Update the table where x = 2
table.update(values_sql={"x": "x + 1"})

print(table.to_pandas())

   x    vector
0  2    [1, 2]
1  4    [5, 6]
2  3  [10, 10]


## Drop a table
Use drop_table() method on the database to remove a table

In [37]:
# Synchronous client
db.drop_table("my_table")
# Asynchronous client
# await async_db.drop_table("my_table2")

### Vector Search
A vector search finds the approximate or exact nearest neighbors to a given query vector.

In a recommendation system or search engine, you can find similar records to the one you searched.
In LLM and other AI applications, each data point can be represented by embeddings generated from existing models, following which the search returns the most relevant features.

#### Distance metrics
Distance metrics are a measure of the similarity between a pair of vectors. Currently, LanceDB supports the following metrics:
<ul>
<li>Metric	Description</li>
<li>l2	Euclidean / L2 distance</li>
<li>cosine	Cosine Similarity</li>
<li>dot	Dot Production</li>
</ul>

### Exhaustive search (kNN)
If you do not create a vector index, LanceDB exhaustively scans the entire vector space and computes the distance to every vector in order to find the exact nearest neighbors. This is effectively a kNN search.

## Disk-based Index
Lance provides an IVF_PQ disk-based index. It uses Inverted File Index (IVF) to first divide the dataset into N partitions, and then applies Product Quantization to compress vectors in each partition.LanceDB  indexing concepts guidehasr more information on how this works.

## Table without an Index

In preparation of the experiment, we will wipe out the table with the index, if it exists.

In [38]:
import lancedb
import numpy as np
uri = "data/sample-vectors"
db = lancedb.connect(uri)
db.drop_table('my_vectors')

In [39]:
import lancedb
import numpy as np
uri = "data/sample-vectors"
db = lancedb.connect(uri)

# Create 10,000 sample vectors
data = [{"vector": row, "item": f"item {i}"}
    for i, row in enumerate(np.random.random((10_000, 1536)).astype('float32'))]

# Add the vectors to a table
tbl = db.create_table("my_vectors", data=data,mode="overwrite")

[2026-08-15T03:48:53Z WARN  lance::dataset::write::insert] No existing dataset at /home/maulik/CS104/Week 14 - Embeddings & Vector DBs/notebooks_dataset/data/sample-vectors/my_vectors.lance, it will be created


Examine the content of the table `my_vectors`

In [40]:
print(tbl.search().limit(4).to_pandas())

                                              vector    item
0  [0.2835325, 0.5422919, 0.75389206, 0.7733459, ...  item 0
1  [0.6149487, 0.62258536, 0.84867305, 0.30649805...  item 1
2  [0.6621832, 0.36076483, 0.8730784, 0.13876724,...  item 2
3  [0.6462792, 0.47517583, 0.053959534, 0.9919343...  item 3


### Exhaustive search (kNN)
If you do not create a vector index, LanceDB exhaustively scans the entire vector space and computes the distance to every vector in order to find the exact nearest neighbors. This is effectively a kNN searhc.
Notice that we are passing the `query` vector directly to the `search()`:

In [41]:
import lancedb
import numpy as np
import timeit

db = lancedb.connect("data/sample-vectors")

tbl = db.open_table("my_vectors")

# Query vector is a new random vector of 1536 elements.
# We pass query vector to search().

%timeit df = tbl.search(query = np.random.random((1536)),vector_column_name ="vector")\
    .limit(10) \
    .to_list()

43 ms ± 676 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [42]:
#print(df[1])

### Search with table index
We will first index table `my_vectors` and then conduct the search.

In [43]:
import timeit

# Create and train the index - you need to have enough data in the table for an effective training step
%timeit tbl.create_index(vector_column_name="vector", metric="cosine",num_partitions=256,num_sub_vectors=96)

<magic-timeit>:1: DeprecationWarning: The create_index() API with metric/num_partitions parameters is deprecated and will be removed in a future version. Please migrate to the new unified API:
  # Old (deprecated):
  table.create_index('l2', vector_column_name='my_vector')
  # New (recommended):
  table.create_index('my_vector', config=IvfPq(distance_type='l2'))


4.86 s ± 952 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [44]:
# The query vector is a new random vector of 1536 elements.
# We pass query vector to search(). The table has an index, now

%timeit df = tbl.search(query = np.random.random((1536)),vector_column_name ="vector")\
    .limit(10) \
    .to_list()

6.04 ms ± 133 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [45]:
# The query vector is a new random vector of 1536 elements.
# We pass query vector to search(). The table has an index, now.
#This time we are not specifying the query and the vector column.

%timeit df = tbl.search(np.random.random((1536)))\
    .limit(10) \
    .to_list()

5.98 ms ± 140 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


### From Pydantic Models
When you create an empty table without data, you must specify the table schema. LanceDB supports creating tables by specifying a PyArrow schema or a specialized Pydantic model called LanceModel.

For example, the following Content model specifies a table with 5 columns: movie_id, vector, genres, title, and imdb_id. When you create a table, you can pass the class as the value of the schema parameter to create_table. The vector column is a Vector type, which is a specialized Pydantic type that can be configured with the vector dimensions. It is also important to note that LanceDB only understands subclasses of lancedb.pydantic.LanceModel (which itself derives from pydantic.BaseModel).

In [47]:
from lancedb.pydantic import Vector, LanceModel

class Content(LanceModel):
    movie_id: int
    vector: Vector(128)
    genres: str
    title: str
    imdb_id: int

    @property
    def imdb_url(self) -> str:
        return f"https://www.imdb.com/title/tt{self.imdb_id}"

import pyarrow as pa
db = lancedb.connect("db/lancedb")
table_name = "movielens_small"
table = db.create_table(table_name, schema=Content, mode = "overwrite")

In [48]:
table.search().limit(10).to_pandas()

,movie_id,vector,genres,title,imdb_id


### OpenAI Embedding function
LanceDB registers the OpenAI embeddings function in the registry as openai. You can pass any supported model name to the `create`. By default, it uses "text-embedding-ada-002".

### OpenAI Embedding function
LanceDB registers the OpenAI embeddings function in the registry as openai. You can pass any supported model name to the `create`. By default it uses "text-embedding-ada-002".

In [50]:
import os
from dotenv import load_dotenv

# Load environment variables from the .env file in your directory
load_dotenv()

# Retrieve the API key from the environment
api_key = os.getenv("OPENAI_API_KEY")

# Verify if the key was loaded successfully
if api_key:
    print(f"✓ API Key loaded successfully. Starts with: {api_key[:7]}...")
else:
    print("✗ API Key not found. Please check your .env file.")

✓ API Key loaded successfully. Starts with: sk-proj...


In [51]:
import lancedb
from lancedb.pydantic import LanceModel, Vector
from lancedb.embeddings import get_registry

db = lancedb.connect("data/db")
func = get_registry().get("openai").create(name="text-embedding-3-small")

class Words(LanceModel):
    text: str = func.SourceField()
    vector: Vector(func.ndims()) = func.VectorField()

table = db.create_table("words", schema=Words, mode="overwrite")
table.add(
    [
        {"text": "The quick brown fox"},
        {"text": "Jumped over the lazy dog"}
    ]
    )

query = "brown"
result = table.search(query).limit(1).to_pydantic(Words)[0]
print(result.text)

The quick brown fox


In [52]:
table.search().to_pandas()

,text,vector
0,The quick brown fox,"[-0.031707764, -0.006286621, 0.0050697327, -0...."
1,Jumped over the lazy dog,"[0.033447266, -0.019165039, -0.026519775, -0.0..."


`func.VectorField()` tells LanceDB to use the OpenAI embedding function to generate query embeddings for the vector column and SourceField ensures that when adding data, we automatically use the specified embedding function to encode text values.